# Plot Training Logs

**Plot the training logs of a model you have trained.** 

This notebook plots the Learning Rate (Eta), the Exploration Rate (Epsilon), the Reward by Episode and the Loss by Episode as a function of the episode.

Command to Pip Install pandas included.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
#Add path to model you have trained
PATH = "../models/0610_0002_CarDQNAgent"
logs = pd.read_csv(os.path.join(PATH,"log.csv"))
logs.tail()

In [ ]:
logs_by_episode = logs.groupby('Episode').sum('Reward').reset_index()
logs_by_episode.head()

In [ ]:
def plot_logs(logs, title):

    logs_by_episode = logs.groupby('Episode').sum('Reward').reset_index()


    fig, axs = plt.subplots(2,2)


    fig.suptitle(title, fontsize=14)


    #Subplot #1
    axs[0,0].plot(logs["Episode"], logs['Eta']);
    axs[0,0].set_ylabel('Eta')
    axs[0,0].set_xlabel('Episode')
    axs[0,0].set_yscale('log')

    #Subplot #2:
    axs[0,1].plot(logs["Episode"], logs['Epsilon']);
    axs[0,1].set_ylabel('Epsilon')
    axs[0,1].set_xlabel('Episode')
    #axs[0,1].set_yscale('log')

    #Subplot #3:
    axs[1,0].plot(logs_by_episode["Episode"], logs_by_episode['Reward']);
    axs[1,0].set_ylabel('Reward by Episode')
    axs[1,0].set_xlabel('Episode')
    #axs[1,0].set_yscale('log')


    #Subplot #4:
    axs[1,1].plot(logs_by_episode["Episode"], logs_by_episode['Loss']);
    axs[1,1].set_ylabel('Loss by Episode')
    axs[1,1].set_xlabel('Episode')
    #axs[1,1].set_yscale('log')

    fig.tight_layout()

In [ ]:
plot_logs(logs, 'Car Racing logs training on 06/06')

#### Rolling avg. reward

In [ ]:
rolling_rewards = logs.groupby('Episode').sum()['Reward'].rolling(30).mean()
ax = rolling_rewards.plot()
ax.set_title('Reward at end of episode (rolling over 30 eps.)')
ax.set_xlabel('Episode n°')
ax.set_ylabel('Reward');

#### Episode length (in steps)

In [ ]:
logs_by_episode = logs.groupby('Episode').count()
ax = logs_by_episode['Time'].plot()
ax.set_title('Episode Length (in steps)')
ax.set_xlabel('Episode n°')
ax.set_ylabel('Episode length');

### Action pdf as a function of episode

In [ ]:
action_count_per_ep = logs[['Episode', 'Action']].groupby('Episode').value_counts(normalize=True).unstack(level=-1)
action_count_per_ep[action_count_per_ep.isna()] = 0
neps = action_count_per_ep.index.max()
fig, ax = plt.subplots()
ax.imshow(np.array(action_count_per_ep).T, aspect=neps//30)
fig.set_size_inches(w=15, h=4)


# # Frozen Lake
# ax.set_yticks([0,1,2,3])
# ax.set_yticklabels(['Left', 'Down', 'Right', 'Up'])

# Car Race
ax.set_yticks([0,1,2,3,4])
ax.set_yticklabels(['Idle', 'Left', 'Right', 'Gas', 'Brake'])

ax.set_xlabel('Episode n°')
ax.set_ylabel('Action taken');

#### Graphing the reward time series for each episode 
(see if at least the model is progressing)

In [ ]:
import matplotlib as mpl
import matplotlib.cm as cm


fig, ax = plt.subplots()
neps = logs['Episode'].max()
cmap = cm.magma

for episode in range(neps):
    rewards_ep = logs[logs['Episode'] == episode]['Reward'].cumsum().to_numpy()
    ax.plot(rewards_ep, color=cmap(episode/neps))

ax.plot([0, 1000], [-8, -8], 'r')
ax.plot([0, 1000], [0, 0], 'k:')

ax.set_xlabel('Step in episode')
ax.set_ylabel('Cumulative Reward')
ax.set_ylim([-10,1000])
fig.set_size_inches(w=13, h=6)

norm = mpl.colors.Normalize(vmin=0, vmax=neps)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
             ax=ax, label='Episode no')